In [ ]:
# WITHIN-SUBJECT
import mne
import torch
from moabb.datasets import Zuo2025

dataset = Zuo2025()
data = dataset.get_data()

subjects = list(data.keys())


def epochs_from_session(session_dict, tmin=0, tmax=4.0):

    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# dicionário: subject_id -> {"train": epochs_sessao0, "test": epochs_sessao1}
within_subject_tensors = {}

for subj in subjects:
    session_keys = list(data[subj].keys())

    train_epochs_0 = epochs_from_session(data[subj][session_keys[0]])
    train_epochs_1 = epochs_from_session(data[subj][session_keys[1]])
    train_epochs_2 = epochs_from_session(data[subj][session_keys[2]])
    test_epochs_3 = epochs_from_session(data[subj][session_keys[3]])
    test_epochs_4 = epochs_from_session(data[subj][session_keys[4]])

    train_list = [train_epochs_0, train_epochs_1, train_epochs_2]
    test_list = [test_epochs_3, test_epochs_4]

    X_tr = train_list.get_data() * 1e6
    y_tr = train_list.events[:, -1]
    X_te = test_list.get_data() * 1e6
    y_te = test_list.events[:, -1]

    min_lbl = y_tr.min()

    within_subject_tensors[subj] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_tr - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_te, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_te - min_lbl, dtype=torch.long),
    }

for subj, tensors in within_subject_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

In [ ]:
#CROS-SUBJECT

import mne
import torch
import numpy as np
from moabb.datasets import BNCI2014_001

dataset = BNCI2014_001()
data = dataset.get_data()

subjects = list(data.keys())


def epochs_from_subject(session_dict, tmin=0, tmax=4.0):
   
    raws = []
    for sess in session_dict.values():
        raws.extend(list(sess.values()))

    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# Extrai os epochs de cada sujeito 
subject_epochs = {}
for subj in subjects:
    subject_epochs[subj] = epochs_from_subject(data[subj])


all_events = np.concatenate([subject_epochs[s].events[:, -1] for s in subjects])
min_lbl = all_events.min()

# Monta um fold para cada sujeito deixado de fora (leave-one-subject-out)
cross_subject_folds = {}

for test_subj in subjects:
    train_subjs = [s for s in subjects if s != test_subj]

    X_train_list, y_train_list = [], []
    for s in train_subjs:
        ep = subject_epochs[s]
        X_train_list.append(ep.get_data() * 1e6)
        y_train_list.append(ep.events[:, -1])

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)

    X_test = subject_epochs[test_subj].get_data() * 1e6
    y_test = subject_epochs[test_subj].events[:, -1]

    cross_subject_folds[test_subj] = {
        "X_train": torch.tensor(X_train, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_train - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_test, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_test - min_lbl, dtype=torch.long),
    }

# iterar sobre os folds, cada um deixando um sujeito de fora
for test_subj, tensors in cross_subject_folds.items():
    X_train_fold = tensors["X_train"]
    y_train_fold = tensors["y_train"]
    X_test_fold = tensors["X_test"]
    y_test_fold = tensors["y_test"]

    print(f"Fold (teste={test_subj}): "
          f"treino={tuple(X_train_fold.shape)}, teste={tuple(X_test_fold.shape)}")